# NB16 — Descriptive statistics (baseline + extended feature sets)

Addresses supervisor feedback **#2**: a descriptive-statistics subsection for the Data/Methodology chapter.

**Scope / decisions**
- Covers **both modelling feature sets**: `baseline` (NB11a) and `extended` (NB11b).
- Described on the **contemporaneous** merged panels (`data/merged/…`, the NB09 output), **not** the lagged modelling matrices (`me_merged_*`). The lag (NB10) shifts every value +1 year and drops each bank's first year, which would mis-date episodes and discard some extremes; distributions are otherwise near-identical. See the SPECS cell for the full rationale.
- Each spec is described **after the same collinearity `DROPS` its notebook applies**, so the table shows only variables the models actually use (baseline drops accounting identities + net loans + total expenses; extended additionally drops `l_loans`, and drops `FM.AST.NFRG.CN` from macro since it only feeds the derived `nfa_gdp`).
- **Micro is summarised at bank-year; macro at country-year (deduplicated).** Macro is constant within a country-year, so describing it on the bank-year panel would weight each country-year by its bank count and distort the means against known EU aggregates. The differing `count` column (~332 vs 4646) reflects the two bases.
- Values are **post data-prep (Layer-1) imputation**; residual NaNs remain (see `count`) because the per-window model-time imputer in NB11 fills the rest.
- **Missingness is reused, not recomputed** (`data/micro/clean/…`, `data/macro/clean/missings/…`). The recorded `missing_pct` (micro before imputation, macro after internal-gap interpolation) is only joined in for reference. Three robust-only micro vars aren't in the coded baseline file, so their missingness is mapped from the Title-Case `missingness_full_comparison.csv`.

**Outputs**
- `outputs/tables/descriptive_statistics.xlsx` (sheets: `baseline`, `extended`) + per-spec `.csv`
- `outputs/figures/descriptives/correlation_matrix_baseline.png` and `…_extended.png`
- `outputs/figures/descriptives/distributions_nobenchmark_baseline.png` and `…_extended.png` (no-GFDD-benchmark micro variables, skew-annotated)
- `outputs/figures/descriptives/distributions_benchmark_micro.png` — benchmarkable micro ratios with the GFDD median line (baseline + extended combined; extended-only vars in amber)
- `outputs/figures/descriptives/distributions_macro.png` — all macro indicators with the IMF WEO median line where an aggregate exists (baseline + extended combined; extended-only vars in amber)

**Still to write (prose):** economic-plausibility discussion + outlier/winsorising note (candidates in the final cell).

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# A58: paths relative to notebooks/, like every other notebook (they were relative to the repository root)
OUT_TAB = Path('../outputs/tables'); OUT_TAB.mkdir(parents=True, exist_ok=True)
OUT_FIG = Path('../outputs/figures') / 'descriptives'; OUT_FIG.mkdir(parents=True, exist_ok=True)

# Readable labels — stripped 'shap_' prefix from the NB15 label dict (single source of
# truth for figure labels); robust-only variables added so the extended table matches.
LABELS = {
    'r_cti': 'Cost-to-income ratio', 'x_loans_gross': 'Gross loans & advances',
    'x_ib_liab': 'Interbank liabilities', 'inc_nii': 'Net interest income',
    'p_nim': 'Net interest margin', 'inc_op': 'Operating income',
    'p_roa': 'ROA', 'p_roe': 'ROE', 'x_dep': 'Customer deposits',
    'c_eqratio': 'Equity ratio', 'l_assets': 'Log total assets',
    'l_loans': 'Log loans', 'r_loans_assets': 'Loans to assets',
    'r_dep_liab': 'Deposits to liabilities', 'c_leverage': 'Leverage (assets/equity)',
    'cr_impchg': 'Impairment charges', 'r_loans_dep': 'Loans/deposits',
    'l_obsi': 'Log off-bal. items',
    'NGDP_RPCH': 'Real GDP growth', 'NGDPD': 'Nominal GDP',
    'PCPIPCH': 'Inflation (CPI)', 'LUR': 'Unemployment rate',
    'BCA_NGDPD': 'Current account (% GDP)', 'BIS_REER': 'Real eff. exchange rate',
    'RXF11FX_REVS': 'Log FX reserves', 'reer_g': 'REER growth',
    'fx_g': 'FX reserves growth', 'FS.AST.PRVT.GD.ZS': 'Private credit (%GDP)',
    'nfa_gdp': 'NFA change (% GDP)', 'credit_g': 'Credit growth', 'credit_gap': 'Credit-to-GDP gap',
    'credit_gap_missing': 'Credit gap unavailable (0/1)',
}

In [ ]:
# --- missingness as recorded by the cleaning notebooks, reused (not recomputed) ------
# A58: micro values are before imputation; the macro file is written after internal-gap
# interpolation, so the column is 'as recorded', not 'pre-imputation'.
miss = pd.concat([
    pd.read_csv('../data/micro/clean/missingness_baseline_2004_2017.csv')
      .set_index('variable')['missing_pct'],
    pd.read_csv('../data/macro/clean/missings/macro_combi_missing_2004_2017.csv')
      .set_index('variable')['missing_pct'],
])
# The coded baseline file already covers cr_impchg and r_loans_dep. Only l_obsi is
# absent (the file lists x_obsi, its pre-log level); l_obsi shares x_obsi's
# missingness (8.674). Drop any duplicate index so miss.get() returns a scalar
# (a duplicated index made it return a Series -> broke those table cells).
miss = pd.concat([miss, pd.Series({'l_obsi': 8.674})])
miss = miss[~miss.index.duplicated(keep='first')]

In [ ]:
# --- spec definitions: mirror NB11a (baseline) / NB11b (extended) variable sets -----
# IMPORTANT: we read the CONTEMPORANEOUS merged panels (`data/merged/…`, the NB09
# output), NOT the lagged modelling matrices (`data/engineered/me_merged_*`,
# which NB10 produces by shifting every feature 1 year within each bank). Reason: the
# descriptive statistics should characterise the variables at their natural calendar
# timing. The lagged files attribute each observation to the *target* year, which shifts
# every value +1 year (e.g. Latvia's 2009 GDP crash lands at 2010) and, because the lag
# drops each bank's first year, also silently discards some extreme observations. The
# distributions are near-identical either way, but the contemporaneous panel gives
# correct episode years and a fuller sample (n≈332 country-years / 4646 bank-years vs
# 306 / 4073). We still mirror each spec's variable set and collinearity DROPS exactly.
SPECS = {
    'baseline': dict(
        path='../data/merged/parquet_saves/merged_base.parquet',
        macro=['NGDP_RPCH', 'NGDPD', 'PCPIPCH', 'LUR', 'BCA_NGDPD',
               'BIS_REER', 'RXF11FX_REVS', 'reer_g', 'fx_g'],
        drop_micro=['x_assets', 'x_equity', 'x_liab', 'x_liab_equity',
                    'x_loans_net', 'exp_total'],
        drop_macro=[]),
    'extended': dict(
        path='../data/merged/parquet_saves/merged_robust.parquet',
        macro=['NGDP_RPCH', 'PCPIPCH', 'LUR', 'BCA_NGDPD', 'BIS_REER', 'RXF11FX_REVS',
               'NGDPD', 'FS.AST.PRVT.GD.ZS', 'FM.AST.NFRG.CN', 'reer_g', 'fx_g',
               'nfa_gdp', 'credit_g', 'credit_gap'],  # A23/A24
        drop_micro=['x_assets', 'x_equity', 'x_liab', 'x_liab_equity',
                    'x_loans_net', 'exp_total', 'l_loans'],
        drop_macro=['FM.AST.NFRG.CN']),
}
id_cols = ['bank_id', 'country_iso', 'year', 'crisis']

def build(spec):
    df = pd.read_parquet(spec['path'])
    macro_cols = [c for c in spec['macro'] if c not in spec['drop_macro']]
    micro_cols = [c for c in df.columns
                  if c not in id_cols + spec['macro'] + spec['drop_micro']]
    # Macro variables are constant within a country-year, but the modelling panel
    # repeats each country-year once per bank. Describing macro on the bank-year panel
    # would therefore weight each country-year by its bank count, distorting the means
    # against known EU aggregates (e.g. real GDP growth is pulled from ~1.8% country-
    # year down to ~1.4% bank-year). So macro is summarised on the DEDUPLICATED
    # country-year frame; micro stays bank-year (each bank-year is a genuine micro
    # observation). The differing `count` column (~332 vs 4646) makes the two bases
    # self-evident in the table.
    df_cy = df.drop_duplicates(['country_iso', 'year'])
    def block(data, cols, level):
        d = data[cols].describe().T.rename(columns={'25%': 'p25', '50%': 'median', '75%': 'p75'})
        d.insert(0, 'level', level)
        d.insert(0, 'indicator', [LABELS.get(c, c) for c in d.index])
        # Fisher skewness (0 = symmetric, >1 = strong right tail). NaNs dropped per
        # column so residual missings don't propagate. Pairs with the distribution-grid
        # figure below for the level variables that have no external (GFDD) benchmark.
        d['skew'] = [data[c].dropna().skew() for c in d.index]
        d['missing_pct_recorded'] = [miss.get(c, np.nan) for c in d.index]
        return d
    num = ['count', 'mean', 'std', 'min', 'p25', 'median', 'p75', 'max', 'skew', 'missing_pct_recorded']
    # micro described on the bank-year panel (df); macro on the country-year frame (df_cy)
    table = pd.concat([block(df, micro_cols, 'Micro'), block(df_cy, macro_cols, 'Macro')])
    table = table[['indicator', 'level'] + num]
    table[num] = table[num].round(3)
    return df, micro_cols, macro_cols, table

In [ ]:
# --- build both tables + write one workbook with a sheet per spec ------------------
built = {}
with pd.ExcelWriter(OUT_TAB / 'descriptive_statistics.xlsx') as xl:
    for name, spec in SPECS.items():
        df, micro_cols, macro_cols, table = build(spec)
        built[name] = (df, micro_cols, macro_cols, table)
        table.to_excel(xl, sheet_name=name)
        table.to_csv(OUT_TAB / f'descriptive_statistics_{name}.csv')
        print(f'[{name}] n={len(df)}  {len(micro_cols)} micro + {len(macro_cols)} macro')

built['extended'][3]   # show the extended table inline

In [ ]:
# --- correlation matrix per spec --------------------------------------------------
# One matrix over micro+macro together (cross-level correlations visible); black
# divider separates the blocks; only |r|>=0.5 cells annotated to stay legible.
# The matrix is symmetric, so we mask the diagonal + upper triangle (np.triu) and show
# only the lower triangle — each pair appears once (no mirror-image duplicates).
def heatmap(df, micro_cols, macro_cols, name):
    feat = micro_cols + macro_cols
    corr = df[feat].corr()
    labels = [LABELS.get(c, c) for c in feat]
    mask = np.triu(np.ones_like(corr.values, dtype=bool))   # True on diagonal + above -> hidden
    M = np.ma.masked_array(corr.values, mask=mask)
    cmap = plt.cm.RdBu_r.copy(); cmap.set_bad('white', 0)   # masked cells render blank
    fig, ax = plt.subplots(figsize=(13, 11))
    im = ax.imshow(M, cmap=cmap, vmin=-1, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    split = len(micro_cols) - 0.5
    ax.axhline(split, color='black', lw=1.0); ax.axvline(split, color='black', lw=1.0)
    # annotate only the visible (lower-triangle) cells with |r|>=0.5
    for i in range(len(feat)):
        for j in range(i):
            r = corr.values[i, j]
            if abs(r) >= 0.5:
                ax.text(j, i, f'{r:.2f}', ha='center', va='center', fontsize=5.5,
                        color='white' if abs(r) > 0.7 else 'black')
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Pearson correlation')
    #ax.set_title(f'Correlation matrix — {name} modelling features ', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_FIG / f'correlation_matrix_{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

for name, (df, micro_cols, macro_cols, _) in built.items():
    heatmap(df, micro_cols, macro_cols, name)

In [ ]:
# --- distribution grid: micro variables WITHOUT a GFDD benchmark ------------------
# GFDD only publishes *ratios*, so the absolute levels (EUR m / logs) and the two
# structural ratios (loans/assets, deposits/liabilities) have no external benchmark
# to check plausibility against. Instead we characterise their *shape*. Payoff: the
# raw levels come out strongly right-skewed while the log-transformed size vars
# (l_assets, l_obsi/l_loans) are ~symmetric — a visual justification for the log.
BENCHMARKED = {'r_cti', 'p_nim', 'p_roa', 'p_roe', 'c_eqratio', 'c_leverage', 'r_loans_dep'}
DIST_COL = '#5B7FA6'   # neutral steel-blue (descriptive; not a model-config colour)

def dist_grid(df, micro_cols, name):
    # no-benchmark set = the spec's micro features minus the ones GFDD can benchmark
    vars_ = [c for c in micro_cols if c not in BENCHMARKED]
    n = len(vars_); ncol = 4; nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(14, 3.1 * nrow))
    axes = axes.ravel()
    for ax, v in zip(axes, vars_):
        s = df[v].dropna()
        sk = s.skew()
        # Clip the *display* range to the 1st–99th percentile so the heavy right tail
        # doesn't crush the visible bulk into one bar. Data itself is untouched — this
        # only sets the histogram x-range.
        lo, hi = s.quantile(0.01), s.quantile(0.99)
        ax.hist(s.clip(lo, hi), bins=40, color=DIST_COL, edgecolor='white', linewidth=0.3)
        ax.set_title(LABELS.get(v, v), fontsize=10, fontweight='bold')
        ax.text(0.97, 0.93, f'skew {sk:.1f}', transform=ax.transAxes, ha='right', va='top',
                fontsize=8, color='#333',
                bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#ccc', lw=0.6))
        # Loans-to-assets is a share -> economically should be <= 1. Its skew is driven
        # by a single implausible outlier (max ~8.94) that the 99th-pct display-clip
        # hides, so the picture and the skew number would otherwise disagree. Annotate
        # the true max so the panel is honest (this is a known winsorising candidate).
        if v == 'r_loans_assets':
            ax.annotate(f'max = {s.max():.2f}  (>1)',
                        xy=(0.97, 0.72), xycoords='axes fraction', ha='right', va='top',
                        fontsize=7.5, color='#b2182b',
                        bbox=dict(boxstyle='round,pad=0.3', fc='#fdecec', ec='#b2182b', lw=0.7))
        ax.tick_params(labelsize=7); ax.set_yticks([])
    for ax in axes[n:]:
        ax.axis('off')
    #fig.suptitle(f'Distributions of {name} micro variables without a GFDD benchmark',
     #            #'(own scale per panel; x-axis clipped to 1st–99th percentile for display)',
      #           fontsize=12, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(OUT_FIG / f'distributions_nobenchmark_{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

for name, (df, micro_cols, macro_cols, _) in built.items():
    dist_grid(df, micro_cols, name)

In [ ]:
# --- distribution grids WITH benchmark median lines (baseline + extended combined) --
# Complements the no-benchmark grid above by plotting the *benchmarkable* variables and
# overlaying a red dashed line at the external benchmark median, so the plausibility
# check is visual, not just tabular. Baseline/shared variables keep the steel-blue fill;
# variables that only enter in the EXTENDED spec are drawn in amber and tagged
# "(extended)" in the panel title, so one figure carries both specs (instead of one per
# spec). Benchmarks are stored in the SAME units as the panel variable (the equity ratio
# is a fraction here, so GFDD 7.05% -> 0.0705; loans/deposits is a ratio, so 119% -> 1.19)
# -- verified against the GFDD and IMF WEO source files.
# Both benchmark sets are computed over 2004-2017 to MATCH the sample window (the merged
# panels span 2004-2017). GFDD micro medians are pooled over the same 25 EU countries
# (codes ei01/ei05/ei06/ei07/si03/si04); WEO macro medians are the IMF EU / euro-area
# aggregate groups. Both were checked to reproduce the 2001-2017 reference numbers first.
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# A58: the GFDD and WEO extracts these medians were read from are not in this repository.
GFDD_MED = {'r_cti': 57.1, 'p_nim': 1.82, 'p_roa': 0.65, 'p_roe': 9.04,
            'c_eqratio': 0.0705, 'c_leverage': 14.18, 'r_loans_dep': 1.19}  # 2004-2017, EU-25
WEO_MED  = {'NGDP_RPCH': 2.00, 'PCPIPCH': 2.05, 'LUR': 9.85, 'BCA_NGDPD': 0.90}  # 2004-2017
REER_BASE = {'BIS_REER': 100.0}   # BIS index construction base; no IMF aggregate for REER

EXT_COL   = '#D08A3E'   # amber: variables that only enter in the extended spec
BENCH_RED = '#b2182b'   # same red as the loans-to-assets outlier annotation above

def combined_grid(data, cols, ext_only, benchmarks, tag_of, title, fname):
    n = len(cols); ncol = 4; nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(14, 3.1 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, v in zip(axes, cols):
        s = data[v].dropna()
        is_ext = v in ext_only
        # Clip only the DISPLAY range to 1st-99th pct so heavy tails don't crush the
        # visible bulk; the underlying data is untouched (skew below uses the full data).
        lo, hi = s.quantile(0.01), s.quantile(0.99)
        ax.hist(s.clip(lo, hi), bins=40, color=EXT_COL if is_ext else DIST_COL,
                edgecolor='white', linewidth=0.3)
        ax.set_title(LABELS.get(v, v) + ('  (extended)' if is_ext else ''),
                     fontsize=10, fontweight='bold', color=EXT_COL if is_ext else 'black')
        ax.text(0.97, 0.93, f'skew {s.skew():.1f}', transform=ax.transAxes, ha='right',
                va='top', fontsize=8, color='#333',
                bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#ccc', lw=0.6))
        b = benchmarks.get(v)
        if b is not None:
            # Ensure the benchmark line stays inside the visible range even if the 1-99
            # display clip would otherwise push it off-panel.
            x0, x1 = ax.get_xlim(); ax.set_xlim(min(x0, b), max(x1, b))
            ax.axvline(b, color=BENCH_RED, lw=1.5, ls='--')
            ax.text(0.03, 0.93, f'{tag_of(v)} {b:g}', transform=ax.transAxes, ha='left',
                    va='top', fontsize=7.5, color=BENCH_RED,
                    bbox=dict(boxstyle='round,pad=0.25', fc='#fdecec', ec=BENCH_RED, lw=0.6))
        ax.tick_params(labelsize=7); ax.set_yticks([])
    for ax in axes[n:]:
        ax.axis('off')
    fig.legend(handles=[Patch(fc=DIST_COL, ec='white', label='Baseline set (shared)'),
                        Patch(fc=EXT_COL, ec='white', label='Extended-set addition'),
                        Line2D([0], [0], color=BENCH_RED, lw=1.5, ls='--',
                               label='External benchmark median')],
               loc='lower center', ncol=3, fontsize=9, frameon=False,
               bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(title, fontsize=12, fontweight='bold')
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    fig.savefig(OUT_FIG / fname, dpi=200, bbox_inches='tight')
    plt.show()

# Extended spec is the superset -> plot from it; "extended-only" = extended minus baseline
_, base_micro, base_macro, _  = built['baseline']
df_e, ext_micro, ext_macro, _ = built['extended']
df_e_cy = df_e.drop_duplicates(['country_iso', 'year'])   # macro basis (matches the table)
ext_only_micro = set(ext_micro) - set(base_micro)
ext_only_macro = set(ext_macro) - set(base_macro)

# (1) micro benchmarkable ratios, with GFDD median lines
combined_grid(df_e, [c for c in ext_micro if c in GFDD_MED], ext_only_micro,
              GFDD_MED, lambda v: 'GFDD',
              'Distributions of micro ratios with a GFDD benchmark — baseline + extended sets\n',
              #'(red dashed = World Bank GFDD EU-25 median 2004–2017; own scale, 1st–99th pct display clip)',
              'distributions_benchmark_micro.png')

# (2) all macro indicators, with IMF WEO median lines where an aggregate exists
combined_grid(df_e_cy, ext_macro, ext_only_macro, {**WEO_MED, **REER_BASE},
              lambda v: 'base' if v in REER_BASE else 'WEO',
              'Distributions of macro indicators — baseline + extended sets\n',
              #'(red dashed = IMF WEO EU / euro-area median 2004–2017; REER line = index base 100; '
              #'own scale, 1st–99th pct display clip)',
              'distributions_macro.png')

## Notes for the write-up (prose still to do)

**Existing outlier treatment (from NB01-02) — NOT winsorising.** A few micro ratios have implausible values *set to NaN* via hard domain bounds (the conservative "set-to-missing" option, not percentile capping):
- `equity_ratio`: <0 or >1.5 → NaN
- `leverage` (assets/equity): <0 or >100 → NaN  (visible: max caps at 99.2)
- `loans/deposits` (`r_loans_dep`): >5 → NaN  (visible: max caps at 4.98)

Macro variables received **no** value treatment (only `inf → NaN`). See `Diagnostic checks.ipynb`.

**Untreated → the visible outliers** (these ratios have no bound rule, so extremes survive — this is what the boxplot panel shows):
- `Cost-to-income ratio`: min −258, max 944 (should be ~0–100%)
- `ROE`: min −915 — extreme negative tail
- `Loans to assets`: max 8.94 (a share, so should be ≤ 1)
- `FX reserves growth` (macro): max 46.7 (≈ 4670% YoY)

So the outlier discussion has two parts: (a) *already-bounded* ratios (equity/leverage/loans-deposits) are clean; (b) these four were left raw and are the winsorising candidates for a possible rerun (see `Future rerun improvements.md`). For the green-light version the stance is explain-and-retain (real distressed/small-bank values; trees robust; LR/MLP standardised).

**Correlation observations** — in both specs the only strong (|r|≥0.5) block is the micro *size* cluster; the macro block is near-orthogonal apart from Nominal GDP × Log FX reserves (~0.53). In the extended set the added credit variables (credit growth, credit-to-GDP gap, private credit) are worth eyeballing for collinearity among themselves.

**Do not recompute missingness here** — it lives in the cleaning notebooks / `data_*/clean/`. The `missing_pct_recorded` column is a reference join only.

## Notes for the write-up (prose still to do)

**Outlier / winsorising candidates** (baseline; extended shares them) — ratios that should be bounded but aren't:
- `Cost-to-income ratio`: min −258, max 944 (should be ~0–100%)
- `ROE`: min −915 — extreme negative tail
- `Loans to assets`: max 8.94 (a share, so should be ≤ 1)
- `FX reserves growth`: max 46.7 (≈ 4670% YoY)

**Correlation observations** — in both specs the only strong (|r|≥0.5) block is the micro *size* cluster; the macro block is near-orthogonal apart from Nominal GDP × Log FX reserves (~0.53). In the extended set the added credit variables (credit growth, credit-to-GDP gap, private credit) are worth eyeballing for collinearity among themselves.

**Do not recompute missingness here** — it lives in the cleaning notebooks / `data_*/clean/`. The `missing_pct_recorded` column is a reference join only.

## Sample composition — bank-year vs country-year

Supports the H3 / interaction discussion: macro indicators are constant within a country-year,
so a micro x macro interaction is identified only from *between* country-year contrasts. This
table reports how far the bank-year sample collapses when reduced to distinct macro states, and
how many of those states carry a crisis.

In [ ]:
# --- sample composition: what a CONDITIONAL effect is identified from ---------------
# Macro indicators are constant within a country-year but the modelling panel repeats each
# country-year once per bank. A micro x macro interaction therefore cannot be identified
# within a country-year at all - only by comparing similar banks under different macro
# states. The effective sample for a conditional effect is thus the number of distinct
# country-year macro states, not the number of bank-years. This cell quantifies that gap.
#
# Both bases are reported because they answer different questions:
#   contemporaneous (merged_*)      - what the descriptive statistics above characterise
#   lagged modelling matrices (me_*) - what the models are actually fitted on (NB10 output,
#                                      each feature shifted within bank, so each bank loses
#                                      its first year(s) and the sample shrinks)
COMPOSITION_SETS = {
    'Contemporaneous panel (descriptives)': {
        'Baseline':            '../data/merged/parquet_saves/merged_base.parquet',
        'Extended':            '../data/merged/parquet_saves/merged_robust.parquet',
    },
    'Modelling matrix (lagged)': {
        'Baseline / lag-1':    '../data/engineered/parquet_saves/me_merged_base.parquet',
        'Extended / lag-1':    '../data/engineered/parquet_saves/me_merged_robust.parquet',
        'Baseline / lag-2':    '../data/engineered/parquet_saves/me_merged_base_lag2.parquet',
        'Extended / lag-2':    '../data/engineered/parquet_saves/me_merged_robust_lag2.parquet',
    },
}

_rows = []
for _base, _members in COMPOSITION_SETS.items():
    for _label, _path in _members.items():
        _d = pd.read_parquet(_path)
        # One row per distinct macro state. Deduplicating on country-year is the same
        # operation the descriptives use for the macro block, and for the same reason.
        _cy = _d.drop_duplicates(['country_iso', 'year'])
        _crisis_years = sorted(_cy.loc[_cy['crisis'] == 1, 'year'].unique())
        _rows.append({
            'Base': _base,
            'Specification': _label,
            'Banks': _d['bank_id'].nunique(),
            'Countries': _d['country_iso'].nunique(),
            'Bank-years': len(_d),
            'Country-years': len(_cy),
            'Bank-years per macro state': round(len(_d) / len(_cy), 1),
            'Crisis bank-years': int(_d['crisis'].sum()),
            'Crisis bank-years (%)': round(_d['crisis'].mean() * 100, 1),
            'Crisis country-years': int(_cy['crisis'].sum()),
            'Crisis country-years (%)': round(_cy['crisis'].mean() * 100, 1),
            'Crisis countries': int(_d.loc[_d['crisis'] == 1, 'country_iso'].nunique()),
            'Crisis years spanned': len(_crisis_years),
            'Crisis year range': f"{int(min(_crisis_years))}-{int(max(_crisis_years))}" if _crisis_years else '-',
        })

composition = pd.DataFrame(_rows).set_index(['Base', 'Specification'])
composition.to_excel(OUT_TAB / 'sample_composition.xlsx')
composition.to_csv(OUT_TAB / 'sample_composition.csv')
display(composition)

# Crisis country-years by calendar year - the concentration matters as much as the count,
# because a conditional effect estimated from a single episode cannot be separated from
# whatever else was specific to that episode.
for _base, _members in COMPOSITION_SETS.items():
    for _label, _path in _members.items():
        _d = pd.read_parquet(_path)
        _cy = _d.drop_duplicates(['country_iso', 'year'])
        _by_year = _cy[_cy['crisis'] == 1].groupby('year').size()
        print(f"{_base} | {_label}: " + ', '.join(f'{int(y)}: {n}' for y, n in _by_year.items()))

## Between-country variance share and label persistence

Two descriptive checks the Discussion leans on when explaining why the micro block predicts
systemic crises.

**Between-country variance share** tests whether the micro features are simply encoding country
identity — if they were, a bank-level feature would be nearly constant within a country and the
apparent micro advantage would be a fixed-effects artefact rather than a banking signal.

**Label persistence** quantifies how far the duration-coded crisis label makes the task one of
identifying an ongoing *state* rather than timing an onset.

In [ ]:
# --- between-country variance share of the micro features ---------------------------
# One-way variance decomposition per feature: what fraction of its total variation lies
# BETWEEN countries rather than within them. A share near 1 would mean the feature barely
# varies between banks in the same country, i.e. it is a country marker wearing a bank-level
# label. A low share means the variation is genuinely cross-sectional across banks.
# Computed on the contemporaneous panel, consistent with the descriptives above.
_var_rows = []
for _name, _spec in SPECS.items():
    _df = pd.read_parquet(_spec['path'])
    _micro = [c for c in _df.columns
              if c not in id_cols + _spec['macro'] + _spec['drop_micro']]
    for _c in _micro:
        _d = _df[['country_iso', _c]].dropna()
        if len(_d) < 2 or _d[_c].std() == 0:
            continue
        _grand = _d[_c].mean()
        _g = _d.groupby('country_iso')[_c].agg(['mean', 'count'])
        # Between-group sum of squares weighted by group size / total sum of squares.
        _between = (_g['count'] * (_g['mean'] - _grand) ** 2).sum()
        _total = ((_d[_c] - _grand) ** 2).sum()
        _var_rows.append({'Spec': _name, 'Code': _c, 'Indicator': LABELS.get(_c, _c),
                          'Between-country share': _between / _total})

var_share = pd.DataFrame(_var_rows)
for _name, _g in var_share.groupby('Spec'):
    print(f"{_name:<9} n={len(_g):<3} median={_g['Between-country share'].median():.3f}  "
          f"max={_g['Between-country share'].max():.3f}  "
          f"features above 0.5: {(_g['Between-country share'] > 0.5).sum()}")
display(var_share.pivot(index=['Code', 'Indicator'], columns='Spec',
                        values='Between-country share').round(3)
        .sort_values('baseline', ascending=False))

# --- label persistence --------------------------------------------------------------
# P(crisis_t == crisis_t-1) on the country-year panel. Restricted to genuinely CONSECUTIVE
# years, so a gap in a country's coverage is not silently treated as a one-year step.
_df = pd.read_parquet(SPECS['baseline']['path'])
_cy = _df.drop_duplicates(['country_iso', 'year']).sort_values(['country_iso', 'year'])
_cy['prev'] = _cy.groupby('country_iso')['crisis'].shift(1)
_cy['prev_year'] = _cy.groupby('country_iso')['year'].shift(1)
_ok = _cy[(_cy['prev'].notna()) & (_cy['year'] - _cy['prev_year'] == 1)]
persistence = (_ok['crisis'] == _ok['prev']).mean()
print(f"\nLabel persistence P(crisis_t == crisis_t-1) = {persistence:.3f}  "
      f"(n = {len(_ok)} consecutive country-year pairs)")

var_share.to_csv(OUT_TAB / 'between_country_variance_share.csv', index=False)
print(f"Saved -> {OUT_TAB / 'between_country_variance_share.csv'}")

In [ ]:
# --- Appendix C table: between-country variance share, publication layout -----------
# The cell above writes the long-form CSV (one row per spec x feature), which is the right
# shape for analysis and the wrong shape for a table. The Discussion now leans on five
# numbers from this decomposition (median, max, none above 0.5, log total assets, and
# deposits-to-liabilities as the most country-clustered feature) to close the objection
# that the micro variables are country dummies in disguise. That is the strongest
# objection to the micro result, so the per-feature figures behind it belong in the
# appendix where a reader can check them instead of only in a CSV in the code folder.
#
# Side-by-side instead of 32 stacked rows: 14 of the 18 features appear in both specs, so
# two numeric columns keep the table to 18 rows AND make the extension's three additions
# and its one dropped variable (l_loans) visible as blanks.
#
# Depends on the cell above having run (uses `var_share`, `persistence` and `_ok`).

appendix_c = (var_share
              .pivot(index=['Code', 'Indicator'], columns='Spec',
                     values='Between-country share')
              .rename(columns={'baseline': 'Baseline', 'extended': 'Extended'})
              .round(3)
              # Sort by the baseline share so the most country-clustered features lead.
              # Extended-only features have no baseline value and fall to the end, which
              # is where they belong: they are what the extension added.
              .sort_values('Baseline', ascending=False, na_position='last')
              .reset_index()
              # Code is dropped for the printed table; Appendix B already maps codes to
              # names, so repeating them here would duplicate rather than reference.
              .drop(columns='Code')
              .set_index('Indicator'))

# The summary line the Discussion quotes directly, exported rather than read off a print.
summary = (var_share.groupby('Spec')['Between-country share']
           .agg(n='size', median='median', max='max').round(3))
summary['above 0.5'] = (var_share[var_share['Between-country share'] > 0.5]
                        .groupby('Spec').size().reindex(summary.index, fill_value=0))

# Label persistence is printed by the cell above but was never exported, so it failed the
# "must sit in an exported table" condition and could not be cited. One row, written
# beside the table it supports because both answer the same question about the label.
#
# WHICH LABEL: the main Laeven & Valencia (2026) `crisis` column. NOT the Reinhart & Rogoff
# arm (separate codebase under Different Definition/Code_RR/) and NOT the pre-crisis
# relabelling (`crisis_rt2`, a different column). Named explicitly because it is the first
# thing a reader will ask, and because this column is identical across both indicator sets
# and both lags, so the figure characterises the LABEL itself, not one specification.
persistence_row = pd.DataFrame([{
    'Statistic': 'P(crisis_t == crisis_t-1)',
    'Value': round(persistence, 3),
    'n (consecutive country-year pairs)': len(_ok),
    'Crisis definition': 'Laeven & Valencia (2026), duration-coded',
    'Panel': 'Contemporaneous country-year panel (baseline spec)',
}])

with pd.ExcelWriter(OUT_TAB / 'appendix_c_between_country_variance.xlsx') as xl:
    appendix_c.to_excel(xl, sheet_name='by_feature')
    summary.to_excel(xl, sheet_name='summary')
    persistence_row.to_excel(xl, sheet_name='label_persistence', index=False)
appendix_c.to_csv(OUT_TAB / 'appendix_c_between_country_variance.csv')
persistence_row.to_csv(OUT_TAB / 'label_persistence.csv', index=False)

display(summary)
display(appendix_c)
print(f"Saved -> {OUT_TAB / 'appendix_c_between_country_variance.xlsx'} (+ 2 csv)")

## Credit-cycle sample composition

Supports the ALE discussion. The ALE curves and the signed SHAP direction both show credit
**below** trend associated with **higher** predicted risk, which runs opposite to the canonical
early-warning reading in which an *above*-trend gap marks the build-up (Drehmann et al., 2011).

This cell asks whether that inversion is simply what the available sample teaches: what did the
credit gap actually look like in the country-years the models were trained and tested on, and do
the data ever contain a boom preceding a bust?

In [ ]:
# --- credit-gap sample composition: what the models could learn about the credit cycle ------
# Motivation: the ALE curves (and the signed SHAP direction, rho = -0.64 in all six models) put
# a BELOW-trend credit gap with HIGHER predicted risk, the opposite of the canonical build-up
# reading. Before treating that as an anomaly, check what the sample contains: if every crisis
# observation available is mid-episode, during deleveraging, then "credit below trend raises
# risk" is the only pattern there was to learn, and the inversion is a property of the target
# and window rather than a misbehaving model.
#
# Two design points, both mirroring the cells above:
#  1. credit_gap is a MACRO variable, constant within a country-year, whilst the panel repeats
#     each country-year once per bank. Taking medians on the bank-year panel would weight every
#     country-year by its bank count (the same distortion the descriptives cell avoids). So we
#     deduplicate to country-year first.
#  2. Both bases are reported, as in the sample-composition cell. The contemporaneous panel
#     dates the gap to the crisis year itself; the lagged modelling matrices are what the models
#     were actually fitted on, where NB10 has shifted every feature within bank, so the gap on a
#     row labelled year t is the value from t-1 (lag-1) or t-2 (lag-2). Those lagged rows are
#     what the ALE curves are computed over, so they are the ones the argument rests on.
#
# NOTE on missing gaps: the one-sided (real-time) HP filter needs a 6-year warm-up, so the first
# years of each country's series are NaN by construction. They are dropped here, and every group
# reports its own n so the coverage is visible rather than assumed.
CREDIT_GAP_SETS = {
    'Contemporaneous panel':    '../data/merged/parquet_saves/merged_robust.parquet',
    'Modelling matrix / lag-1': '../data/engineered/parquet_saves/me_merged_robust.parquet',
    'Modelling matrix / lag-2': '../data/engineered/parquet_saves/me_merged_robust_lag2.parquet',
}
# 2010-2012 are the test years of the extended lag-1 specification, i.e. the window the ALE
# curves and the signed-direction correlations are computed over. 2005-2007 is the pre-GFC
# build-up: the period in which an above-trend gap would have to appear if the models were ever
# to see a boom precede a bust.
GAP_WINDOWS = {
    'Test window 2010-2012': [2010, 2011, 2012],
    'Build-up 2005-2007':    [2005, 2006, 2007],
    'Full panel':            None,
}

_gap_rows, _gap_year_rows = [], []
for _label, _path in CREDIT_GAP_SETS.items():
    _d = pd.read_parquet(_path)
    # one row per macro state; credit_gap is identical across the banks of a country-year
    _cy = _d.drop_duplicates(['country_iso', 'year'])[
        ['country_iso', 'year', 'crisis', 'credit_gap']]
    for _wname, _yrs in GAP_WINDOWS.items():
        _w = _cy if _yrs is None else _cy[_cy['year'].isin(_yrs)]
        _w = _w.dropna(subset=['credit_gap'])
        for _flag in (0, 1):
            # iterate 0/1 explicitly instead of grouping, so a window with NO crisis
            # observations still yields a visible row with n = 0 rather than vanishing from
            # the table - that absence is the point of the build-up window
            _s = _w.loc[_w['crisis'] == _flag, 'credit_gap']
            _gap_rows.append({
                'Base': _label,
                'Window': _wname,
                'Label': 'Crisis' if _flag else 'Non-crisis',
                'Country-years': len(_s),
                'Median gap': round(_s.median(), 2) if len(_s) else np.nan,
                'Mean gap': round(_s.mean(), 2) if len(_s) else np.nan,
            })
    # Year by year inside the test window. If the crisis countries are deleveraging further each
    # year, rather than sitting at a fixed distance below trend, the two medians should diverge
    # over 2010-2012 - which would make the inversion a trend in the data, not a single offset.
    _t = _cy[_cy['year'].isin(GAP_WINDOWS['Test window 2010-2012'])].dropna(subset=['credit_gap'])
    for _y, _g in _t.groupby('year'):
        _c, _n = _g[_g['crisis'] == 1], _g[_g['crisis'] == 0]
        _gap_year_rows.append({
            'Base': _label, 'Year': int(_y),
            'Crisis n': len(_c),
            'Crisis median gap': round(_c['credit_gap'].median(), 2) if len(_c) else np.nan,
            'Non-crisis n': len(_n),
            'Non-crisis median gap': round(_n['credit_gap'].median(), 2) if len(_n) else np.nan,
        })

credit_gap_composition = pd.DataFrame(_gap_rows).set_index(['Base', 'Window', 'Label'])
credit_gap_by_year = pd.DataFrame(_gap_year_rows).set_index(['Base', 'Year'])
display(credit_gap_composition)
display(credit_gap_by_year)

# Headline numbers for the Discussion, read off the lag-1 modelling matrix because those are the
# rows the ALE curves are computed on.
_h = credit_gap_composition.loc['Modelling matrix / lag-1']
_ct, _nt = _h.loc[('Test window 2010-2012', 'Crisis')], _h.loc[('Test window 2010-2012', 'Non-crisis')]
_cb, _nb = _h.loc[('Build-up 2005-2007', 'Crisis')], _h.loc[('Build-up 2005-2007', 'Non-crisis')]
print(f"\nTest window 2010-2012 : crisis median gap {_ct['Median gap']} (n={int(_ct['Country-years'])})"
      f"  vs  non-crisis {_nt['Median gap']} (n={int(_nt['Country-years'])})")
print(f"Build-up 2005-2007    : crisis country-years = {int(_cb['Country-years'])}"
      f"  |  non-crisis median gap {_nb['Median gap']} (n={int(_nb['Country-years'])})")

credit_gap_composition.to_csv(OUT_TAB / 'credit_gap_sample_composition.csv')
credit_gap_by_year.to_csv(OUT_TAB / 'credit_gap_by_year.csv')
print(f"Saved -> {OUT_TAB / 'credit_gap_sample_composition.csv'}")
print(f"Saved -> {OUT_TAB / 'credit_gap_by_year.csv'}")